In [3]:
# Reveal.js
from notebook.services.config import ConfigManager
cm = ConfigManager()
cm.update('livereveal', {
        'theme': 'white',
        'transition': 'none',
        'controls': 'false',
        'progress': 'true',
})

{'theme': 'white',
 'transition': 'none',
 'controls': 'false',
 'progress': 'true'}

In [2]:
%%html
<script>
  function code_toggle() {
    if (code_shown){
      $('div.input').hide('500');
      $('#toggleButton').val('Show Code')
    } else {
      $('div.input').show('500');
      $('#toggleButton').val('Hide Code')
    }
    code_shown = !code_shown
  }

  $( document ).ready(function(){
    code_shown=false;
    $('div.input').hide()
  });
</script>
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>

In [3]:
import random
from IPython.display import Image

# Word and Sentence Embeddings


## Learning objectives

By the end of this lecture, you should be able to:

- Compare one-hot, count-based and dense static word representations
- Use cosine similarity and explain what distributional learning captures
- Explain why one vector per word cannot represent meaning in context
- Distinguish pooled word vectors from Transformer-based sentence embeddings


# Feed-forward Neural Networks
<center><img src="../img/mlp.svg"></center>

## Why representations matter

A model cannot operate directly on words. A representation decides which distinctions are easy for the model to use.

Today we move from word identities to vectors learned from context, then preview representations of whole sentences.


## What makes a representation useful?

- Different inputs need distinguishable vectors.
- Inputs that behave similarly in the task should have nearby vectors.
- The right similarity depends on the task and training data.

A representation keeps some information and discards the rest.


## Formal Task ##

* Words: $w$
* Vocabulary: $\mathbb{V} (\forall_{i} w_{i} \in \mathbb{V})$
* Find representation function: $f(w_{i}) = r_{i}$

## One-hot word representations

Give each vocabulary item its own dimension:

$$f(w) \in \{0,1\}^{|V|}$$

One-hot vectors preserve word identity, but every pair of different words is equally dissimilar and the vectors grow with the vocabulary.


### Example ###

* $\mathbb{V} = \{\textrm{apple}, \textrm{orange}, \textrm{rabbit}\}$
* $f_{id}(\textrm{apple}) = 1, \ldots$, $f_{id}(\textrm{rabbit}) = 3$
* $f_{sb}(\textrm{apple}) = (1, 0, 0)$
* $f_{sb}(\textrm{orange}) = (0, 1, 0)$
* $f_{sb}(\textrm{rabbit}) = (0, 0, 1)$

## Sparse Binary Visualised ##

![Sparse binary representations visualised](../img/sparse_binary.svg)


## Cosine similarity

For vectors $u$ and $v$:

$$\cos(u,v)=\frac{u\cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

Cosine similarity compares direction rather than magnitude. It is useful only when the geometry of the representation has been learned for a relevant objective.


Note the different formulation in SciPy:
$$cos(u, v) = 1 - \frac{u \cdot v}{||u|| \cdot||v||}$$

In [5]:
Image(url='../img/quiz_time.png'+'?'+str(random.random()))

## [tinyurl.com/diku-nlp-cos](https://tinyurl.com/diku-nlp-cos)
([Responses](https://docs.google.com/forms/d/1mbVCSbDXpA9qY5DkfR-fnX3c3MFXPiLAqeTrd2P0U90/edit#responses))

## Dense static word embeddings

Store one learned $d$-dimensional vector for each vocabulary item:

$$W \in \mathbb{R}^{|V| \times d}, \qquad f(w)=W_{w,:}$$

Dense vectors can place related words near one another, but every occurrence of a word receives the same vector.


### Example ###

* $\mathbb{V} = \{\textrm{apple}, \textrm{orange}, \textrm{rabbit}\}$
* $d = 2$
* $W \in \mathbb{R}^{3 \times 2}$
* $f_{id}(\textrm{apple}) = 1, \ldots, f_{id}(\textrm{rabbit}) = 3$
* $f_{dc}(\textrm{apple}) = (1.0, 1.0)$
* $f_{dc}(\textrm{orange}) = (0.9, 1.0)$
* $f_{dc}(\textrm{rabbit}) = (0.1, 0.5)$

## Dense word embeddings in two dimensions

<center><img src="../img/dense_continuous.svg" width="80%"></center>

## Similarity in dense space

* $cos(f_{dc}(\textrm{apple}),f_{dc}(\textrm{rabbit})) \approx 0.83$
* $cos(f_{dc}(\textrm{apple}),f_{dc}(\textrm{orange})) \approx 1.0$
* $cos(f_{dc}(\textrm{orange}),f_{dc}(\textrm{rabbit})) \approx 0.86$

# Learning word embeddings from text

## Learning from unlabelled text

Text supplies its own training signal: nearby words help predict one another.

The language-modelling idea from the first half of today therefore also gives us a way to learn representations without task labels.


## The distributional hypothesis

Words that occur in similar contexts often have related meanings.

> You shall know a word by the company it keeps.

Firth (1957)

This is an empirical shortcut, not a complete theory of meaning: corpus choice and social patterns shape the resulting space.


## Co-occurrence examples

Build the matrix from raw text such as Wikipedia, news or web pages.

1. "…comparing an **apple** to an **orange**…"
2. "…an **apple** and **orange** from Florida…"
3. "…my **rabbit** is not shaped like an **orange**…"


## Count-based context vectors

Count how often each target word occurs near each context word. A row of the co-occurrence matrix becomes the word representation:

$$C \in \mathbb{N}^{|V| \times |V|}, \qquad f(w)=C_{w,:}$$

Words with similar rows appeared in similar contexts. Weighting and dimensionality reduction can improve these raw counts.


### Example ###

* $\mathbb{V} = \{\textrm{apple}, \textrm{orange}, \textrm{rabbit}\}$
* $f_{id}(\textrm{apple}) = 1, \ldots, f_{id}(\textrm{rabbit}) = 3$
* $f_{cs}(\textrm{apple}) = (2, 2, 0)$
* $f_{cs}(\textrm{orange}) = (2, 3, 1)$
* $f_{cs}(\textrm{rabbit}) = (0, 1, 1)$

## Counts and TF–IDF for documents

For text classification, a vector can count each word in a document: $x_{d,w}=\operatorname{count}(w,d)$. Common words can then dominate the vector.

TF–IDF reduces the weight of words that occur in many documents. Scikit-learn's default smoothed inverse-document frequency is:

$$\operatorname{tfidf}(w,d)=\operatorname{count}(w,d)\left[\log\frac{1+N}{1+\operatorname{df}(w)}+1\right]$$

Lab 3 compares raw counts and TF–IDF with the same linear classifier.

# Static neural word embeddings


## Learning by context prediction

> I had some **_____** for breakfast today.

A model should score *cereal* above *airplanes*. Training this prediction repeatedly makes words with similar contexts acquire similar vectors.


## Word2vec predicts neighbouring words

<center><img width=1000 src="../img/cbow_sg2.png"></center>

<div style="text-align: right;">
    (word2vec: <a href="https://arxiv.org/abs/1301.3781">Mikolov et al., 2013</a>)
</div>

## Training pairs from a context window

From “I had some cereal for breakfast”, a small window produces pairs such as:

- target *cereal*, context *some*
- target *cereal*, context *for*
- target *breakfast*, context *for*

Word2vec learns to score observed pairs above sampled non-pairs. The resulting vectors are useful after the prediction model itself is discarded.


## Learned word embeddings in two dimensions

In [6]:
Image(url='../img/word_representations.svg'+'?'+str(random.random()), width=1200)

Nearby points often share syntactic or semantic behaviour, but a two-dimensional t-SNE plot can distort global distances. Test claims in the original embedding space.


## Static word embeddings have limits

The word *bank* receives the same vector in “river bank” and “bank loan”. Word order is also absent from a single lookup vector.

The next two lectures introduce models that build a new representation from the surrounding text.


## A simple sentence representation

A transparent baseline averages the static word vectors:

$$f(s)=\frac{1}{|s|}\sum_{w \in s} f(w)$$

It is fast and often useful, but largely loses word order and treats a word similarly in every context. Lab 3 compares simple representations under the same classifier.


## Three levels of representation

- **Static word embedding:** one vector per vocabulary item.
- **Contextual token embedding:** a different vector for each token occurrence.
- **Sentence embedding:** one vector for a complete sentence or passage.

Next week, RNNs build contextual states. In Week 40, Transformers build contextual token vectors with attention.


## Sentence Transformers

A Sentence Transformer uses a pretrained Transformer to encode a complete text as a fixed-size vector. Models can be trained so cosine similarity reflects sentence-level meaning.

Common uses include semantic search, clustering and classification. We use this as a preview. Week 40 explains the Transformer architecture.


## Sentence embeddings in code

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
sentences = [
    "The lecture starts at ten.",
    "Class begins at 10:00.",
    "The harbour is windy today.",
]
vectors = model.encode(sentences)
```

The same cosine calculation now compares whole sentences.


The example follows the [Sentence Transformers quickstart](https://www.sbert.net/docs/quickstart.html). Here we use the library as a black box. The later lectures explain how it produces contextual vectors.


## Summary

- One-hot vectors preserve identity but express no graded similarity.
- Raw counts and TF–IDF give transparent sparse document representations. Word2vec learns dense static word embeddings from context.
- Cosine similarity measures geometry, whose meaning depends on the training objective.
- Static embeddings ignore the current context; RNNs and Transformers address this limitation.
- Mean pooling is a useful sentence baseline, while Sentence Transformers learn sentence-level vectors for similarity and retrieval.


## Additional reading

- Jurafsky & Martin, [Chapter 5: Embeddings](https://web.stanford.edu/~jurafsky/slp3/5.pdf)
- Mikolov et al. (2013), [Distributed Representations of Words and Phrases and their Compositionality](https://papers.nips.cc/paper_files/paper/2013/hash/9aa42b31882ec039965f3c4923ce901b-Abstract.html)
- Reimers and Gurevych (2019), [Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks](https://aclanthology.org/D19-1410/)
- [Sentence Transformers quickstart](https://www.sbert.net/docs/quickstart.html)
